# Wheat Irrigation Decision Tool - Karnal, Haryana**Goal:** predict tomorrow's reference evapotranspiration (ET0, the crop's water loss in mm)from today's weather, and turn that prediction into an irrigate / don't-irrigate call forwheat growers near Karnal, Haryana (29.69 N, 76.99 E).**This notebook currently covers data setup only:** download two independent weather sources,merge them, and audit the result. No model is built yet.

## Cell 1 - ImportsEverything this notebook needs. We deliberately keep the dependency list tiny (see`requirements.txt`) so the pipeline is easy to reproduce and easy to explain out loud:`requests` to call the two weather APIs, `pandas` / `numpy` to shape the tables, and`matplotlib` to look at the series.

In [ ]:
import requests                   # calls the two keyless weather APIsimport pandas as pd               # all tabular workimport numpy as np                # NaN handling, numeric helpersimport matplotlib.pyplot as plt   # the two diagnostic plots in Cell 5# Show every column when we print wide DataFrames, so nothing hides behind "...".pd.set_option("display.max_columns", 50)pd.set_option("display.width", 200)# Location and period are defined ONCE and reused by both download cells, so the two# sources can never accidentally disagree about what we asked for.LAT, LON   = 29.69, 76.99                  # Karnal, HaryanaSTART, END = "2018-11-01", "2026-03-31"    # start of 2018-19 wheat season -> end of 2025-26print("pandas", pd.__version__, "| numpy", np.__version__)print(f"Location: {LAT} N, {LON} E   Period: {START} to {END}")

## Cell 2 - Download Open-Meteo (primary source, no API key)Open-Meteo's Historical Weather API serves the ERA5 reanalysis. It is our **primary** sourcebecause it already publishes `et0_fao_evapotranspiration`, which is ET0 computed with theFAO-56 Penman-Monteith equation. That is exactly our prediction target. No API key is needed.The request asks for all seven daily variables at once. If the API rejects the request becauseone variable name is invalid, we do **not** silently drop anything: we print the API's realerror message, then re-request the variables one at a time so we can name exactly which onefailed and keep every variable that did work.

In [ ]:
OM_URL = "https://archive-api.open-meteo.com/v1/archive"# The seven daily variables we want. The first one is our prediction target.OM_DAILY = [    "et0_fao_evapotranspiration",   # TARGET: reference evapotranspiration, mm/day (FAO-56)    "precipitation_sum",            # mm/day - rain can make irrigation unnecessary    "temperature_2m_max",           # degC    "temperature_2m_min",           # degC    "wind_speed_10m_max",           # km/h - wind drives evaporative demand    "shortwave_radiation_sum",      # MJ/m2/day - incoming solar energy    "relative_humidity_2m_mean",    # % - dry air raises ET0]def fetch_open_meteo(daily_vars):    # Request a list of daily variables from Open-Meteo.    # Returns (DataFrame, units_dict) on success, and raises RuntimeError on any API error    # so the caller decides what to do. We never return partial data silently.    resp = requests.get(OM_URL, params={        "latitude":   LAT,        "longitude":  LON,        "start_date": START,        "end_date":   END,        "timezone":  "Asia/Kolkata",         # local time, so a "day" matches a farmer's day        "daily":     ",".join(daily_vars),   # the API wants one comma-separated string    }, timeout=180)    # Open-Meteo returns a JSON body with a "reason" field even on errors, so we read the    # body BEFORE raising. That message is what tells us which variable name was rejected.    payload = resp.json()    if resp.status_code != 200:        raise RuntimeError(f"HTTP {resp.status_code}: {payload.get('reason', payload)}")    return pd.DataFrame(payload["daily"]), payload.get("daily_units", {})try:    om, om_units = fetch_open_meteo(OM_DAILY)    print(f"All {len(OM_DAILY)} variables accepted by the API.")except RuntimeError as err:    # Fallback path: the batch request failed. Print the real error, then probe each    # variable on its own so we report precisely which name the API rejected.    print("BATCH REQUEST FAILED - Open-Meteo said:")    print(f"   {err}")    print("\nRetrying one variable at a time to identify the culprit...\n")    accepted, rejected, frames = [], {}, []    for var in OM_DAILY:        try:            one_var, _ = fetch_open_meteo([var])            frames.append(one_var.set_index("time")[[var]])            accepted.append(var)            print(f"   OK        {var}")        except RuntimeError as single_err:            rejected[var] = str(single_err)            print(f"   REJECTED  {var}  ->  {single_err}")    if not accepted:        raise RuntimeError("Open-Meteo rejected every variable; cannot continue.")    om = pd.concat(frames, axis=1).reset_index()   # stitch the surviving columns back together    om_units = {}    print(f"\nKept {len(accepted)} variables; {len(rejected)} rejected and listed above.")    print("Rejected variables are NOT replaced with filled data - they are simply absent.")# Convert the ISO date strings the API returns into real datetimes. This is what lets us# slice by month/season later, and makes the plot x-axis a time axis rather than text.om["time"] = pd.to_datetime(om["time"])print("\nUnits reported by the API:")for name, unit in om_units.items():    print(f"   {name}: {unit}")print(f"\nom shape: {om.shape}   range: {om['time'].min().date()} to {om['time'].max().date()}")om.head()

## Cell 3 - Download NASA POWER (independent cross-check, no API key)NASA POWER is a **different** dataset (MERRA-2 based) produced by a different agency. We pullthe same kind of weather variables so we can later check whether the two sources agree. If theydisagree badly on a day, that tells us our inputs are uncertain for that day, which is exactlythe kind of failure case we want to surface rather than hide.POWER does not supply ET0 in this request, so it is a check on the *inputs*, not on the target.Its missing-data code is `-999`, which we convert to `NaN`. A `-999` left in place would betreated as a real measurement and would silently wreck any average or model fit.

In [ ]:
NASA_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"NASA_PARAMS = [    "T2M_MAX",            # degC      - daily max temperature    "T2M_MIN",            # degC      - daily min temperature    "RH2M",               # %         - relative humidity at 2 m    "WS2M",               # m/s       - wind at 2 m (Open-Meteo gives km/h at 10 m: different!)    "ALLSKY_SFC_SW_DWN",  # MJ/m2/day - all-sky shortwave radiation at the surface    "PRECTOTCORR",        # mm/day    - bias-corrected total precipitation]resp = requests.get(NASA_URL, params={    "parameters": ",".join(NASA_PARAMS),    "community":  "AG",                    # AG = agroclimatology preset    "latitude":   LAT,    "longitude":  LON,    "start": START.replace("-", ""),       # POWER wants YYYYMMDD, not YYYY-MM-DD    "end":   END.replace("-", ""),    "format": "JSON",}, timeout=180)# Fail loudly rather than continuing with a partial or empty table.if resp.status_code != 200:    raise RuntimeError(f"NASA POWER HTTP {resp.status_code}: {resp.text[:500]}")payload = resp.json()# Any warnings POWER attaches (for example about data latency) are worth seeing, not hiding.for message in payload.get("messages", []):    print("NASA POWER message:", message)# POWER nests data as properties.parameter -> {VARNAME: {"20181101": value, ...}}.# Building a DataFrame from that dict puts dates on the index and variables in columns.nasa = pd.DataFrame(payload["properties"]["parameter"])nasa.index = pd.to_datetime(nasa.index, format="%Y%m%d")nasa.index.name = "time"# Count the -999s BEFORE replacing them, so the audit in Cell 5 reflects genuine gaps# instead of us having quietly made them disappear.print("\nCount of -999 (NASA missing-data code) per column, before replacement:")print((nasa == -999).sum().to_string())nasa = nasa.replace(-999, np.nan)print(f"\nnasa shape: {nasa.shape}   range: {nasa.index.min().date()} to {nasa.index.max().date()}")nasa.head()

## Cell 4 - Merge the two sourcesWe join on date with an **inner** join, so `df` holds only days where *both* sources reported.That keeps the cross-check honest: every row can be compared source against source.NASA columns get a `_nasa` suffix. This matters because both sources carry a max temperature, ahumidity, a wind speed and a precipitation. Without the suffix we could not tell which numbercame from which agency, and the whole point of having a second source would be lost.

In [ ]:
# Suffix every NASA column so its provenance is visible in the column name itself.nasa_suffixed = nasa.add_suffix("_nasa")# Open-Meteo is the primary table, so move its date column to the index to join against.om_indexed = om.set_index("time")# Inner join: keep only dates present in BOTH sources (see the markdown above).df = om_indexed.join(nasa_suffixed, how="inner")# Report what the join cost us rather than assuming it was lossless.print(f"om rows:     {len(om_indexed)}")print(f"nasa rows:   {len(nasa_suffixed)}")print(f"merged rows: {len(df)}")print(f"dates in om but not nasa:   {len(om_indexed) - len(df)}")print(f"dates in nasa but not om:   {len(nasa_suffixed) - len(df)}")df = df.sort_index()          # chronological order; required once we start lagging variablesdf.index.name = "date"# Cell 5 adds a derived "season" column to df. Dropping it here keeps this cell safe to# re-run in any order: the CSV always holds raw downloaded data only, never derived columns.df = df.drop(columns=["season"], errors="ignore")# Save inside the project folder only. index=True keeps the date in the CSV.OUT_PATH = "data/karnal_daily.csv"df.to_csv(OUT_PATH, index=True)print(f"\nSaved {df.shape[0]} rows x {df.shape[1]} columns to {OUT_PATH}")print("\nColumns:")for col in df.columns:    print("  ", col)df.head()

## Cell 5 - Data checkThis is the cell that decides whether the project is viable, so we print everything a scepticalreader would ask for:1. **Row count and the date range actually returned**, not the range we asked for. APIs can   quietly return less, because reanalysis data lags real time by days to weeks.2. **Missing values per column.** Any column with gaps is a column we cannot blindly feed a model.3. **Rows per wheat season (Nov-Mar), season by season.** A short season would mean a thin or   unrepresentative test set, which would make the eventual model-vs-baseline comparison misleading.4. **`describe()` on ET0.** A scale sanity check: for this region ET0 should sit roughly between   0.5 and 10 mm/day. Values outside that would point to a unit or parsing bug.Then two plots: the full series, to spot the annual cycle and any flat or broken stretches, andone single wheat season, to see the within-season shape we actually have to predict.

In [ ]:
# ---- 1. Row count and the REAL date range returned --------------------------------print("=" * 70)print("1. ROWS AND DATE RANGE")print("=" * 70)print(f"Requested: {START} to {END}")print(f"Returned:  {df.index.min().date()} to {df.index.max().date()}")print(f"Total rows: {len(df)}")# A complete daily series has exactly one row per calendar day. Comparing the row count with# the number of days in the range shows whether any dates are missing ENTIRELY, which is a# different problem from present-but-NaN (covered in section 2).expected_days = (df.index.max() - df.index.min()).days + 1print(f"Calendar days in that range: {expected_days}")print(f"Missing calendar dates: {expected_days - len(df)}")print(f"Duplicate dates: {int(df.index.duplicated().sum())}")# ---- 2. Missing values per column --------------------------------------------------print("\n" + "=" * 70)print("2. MISSING VALUES PER COLUMN")print("=" * 70)missing = df.isna().sum()print(pd.DataFrame({    "missing": missing,    "pct": (100 * missing / len(df)).round(2),}).to_string())if missing.sum() == 0:    print("\nNo missing values anywhere.")else:    print("\nColumns with gaps:", list(missing[missing > 0].index))# ---- 3. Rows per wheat season ------------------------------------------------------print("\n" + "=" * 70)print("3. ROWS PER WHEAT SEASON (Nov-Mar)")print("=" * 70)def wheat_season(date):    # Label the Nov-Mar wheat season a date belongs to, or None if it is out of season.    # A season spans two calendar years: Nov-Dec sit in the starting year and Jan-Mar in    # the following one, so Nov 2019 and Feb 2020 both belong to season "2019-20".    if date.month >= 11:                       # Nov, Dec -> season starts this year        return f"{date.year}-{str(date.year + 1)[2:]}"    if date.month <= 3:                        # Jan, Feb, Mar -> season started last year        return f"{date.year - 1}-{str(date.year)[2:]}"    return None                                # Apr-Oct: outside the wheat seasondf["season"] = [wheat_season(d) for d in df.index]in_season = df[df["season"].notna()]print(f"Rows inside Nov-Mar wheat seasons: {len(in_season)} of {len(df)} total")print(f"Rows outside (Apr-Oct):            {len(df) - len(in_season)}\n")# A full Nov 1 - Mar 31 season is 151 days, or 152 in a leap year. The "complete?" column# therefore shows at a glance whether a season is truncated at either end.season_counts = in_season.groupby("season").agg(    rows=("season", "size"),    first_date=("season", lambda s: s.index.min().date()),    last_date=("season",  lambda s: s.index.max().date()),)season_counts["complete?"] = np.where(season_counts["rows"] >= 151, "yes", "SHORT")print(season_counts.to_string())# ---- 4. describe() for the ET0 target ----------------------------------------------print("\n" + "=" * 70)print("4. ET0 TARGET SUMMARY  (mm/day)")print("=" * 70)ET0 = "et0_fao_evapotranspiration"print("Full year (all 12 months):")print(df[ET0].describe().to_string())print("\nWheat season only (Nov-Mar), the range the model actually has to handle:")print(in_season[ET0].describe().to_string())# ---- 5. Plots -----------------------------------------------------------------------# Plot A: the whole series. We expect a clean repeating annual wave: low ET0 in the cool# winter, high in the pre-monsoon heat. A flat line or a gap would mean bad data.fig, ax = plt.subplots(figsize=(14, 4))ax.plot(df.index, df[ET0], linewidth=0.7, color="#1f6f8b")ax.set_title("Daily ET0 at Karnal - full series (Open-Meteo / ERA5)")ax.set_xlabel("Date")ax.set_ylabel("ET0 (mm/day)")ax.grid(alpha=0.3)plt.tight_layout()plt.show()# Plot B: one single wheat season, so the within-season shape is visible at this scale.# We pick the most recent COMPLETE season so the picture is not a truncated one.complete_seasons = season_counts[season_counts["complete?"] == "yes"].index.tolist()focus_season = complete_seasons[-1] if complete_seasons else season_counts.index[-1]one = df[df["season"] == focus_season]fig, ax = plt.subplots(figsize=(14, 4))# Rainfall is plotted alongside ET0 because the irrigate decision depends on both:# high water loss only matters if rain has not already supplied the water.ax.bar(one.index, one["precipitation_sum"], color="#5a8fc4", alpha=0.55, label="precipitation (mm)")ax.plot(one.index, one[ET0], marker=".", markersize=3, linewidth=1.0, color="#b8562a", label="ET0 (mm/day)")ax.set_title(f"Daily ET0 and rainfall - wheat season {focus_season} ({len(one)} days)")ax.set_xlabel("Date")ax.set_ylabel("mm/day")ax.legend()ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## Cell 6 - Sources and known limitations### Data sources- **Open-Meteo Historical Weather API** (ERA5 reanalysis), primary source, keyless.  Docs: https://open-meteo.com/en/docs/historical-weather-api  Endpoint used: `https://archive-api.open-meteo.com/v1/archive`  Supplies our prediction target `et0_fao_evapotranspiration` plus the daily weather drivers.- **NASA POWER daily point API** (`community=AG`, MERRA-2 based), independent cross-check, keyless.  Docs: https://power.larc.nasa.gov/docs/services/api/temporal/daily/  Endpoint used: `https://power.larc.nasa.gov/api/temporal/daily/point`  Supplies a second, independently produced estimate of temperature, humidity, wind, radiation  and rainfall for the same point and the same days.### Limitation we must state in the write-up**ET0 here is CALCULATED, not measured.** Both datasets are model reanalysis products. The ET0figure comes from the FAO-56 Penman-Monteith equation applied to modelled gridded weather, notfrom a lysimeter or flux tower standing in a Karnal field. Our model therefore learns to predicta *physics-based estimate* of crop water loss, not ground truth.Three consequences we should be upfront about:1. Any systematic bias in the reanalysis is baked into our target, and our accuracy scores   cannot detect it. We can measure agreement with the estimate, not agreement with reality.2. ERA5 and MERRA-2 are gridded at roughly 10-30 km, so one value stands in for a wide area.   Real field conditions (soil type, irrigation history, local wind shelter) vary inside that cell.3. ET0 is *reference* ET, the demand of a standard short grass surface. Actual wheat water use is   ET0 multiplied by a crop coefficient (Kc) that changes across growth stages. Our output is a   water-demand signal, not a literal "mm of water to apply" figure.Keeping the second source at least lets us quantify how much two independent reanalyses disagreeon the same day, which gives a lower bound on input uncertainty.